In [180]:
# packages
import pandas as pd
import importlib #in order to reload the file
import mod02_build_bot_predictor
importlib.reload(mod02_build_bot_predictor)
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [181]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [182]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [183]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [184]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [185]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [186]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [187]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 121,
 'tn': 2586,
 'fp': 51,
 'fn': 242,
 'misclassification_rate': 0.09766666666666667,
 'false_positive_rate': 0.019340159271899887,
 'false_negative_rate': 0.6666666666666666}

### Check results on the test set (new data not yet seen by the model)

In [188]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 34,
 'tn': 857,
 'fp': 17,
 'fn': 92,
 'misclassification_rate': 0.109,
 'false_positive_rate': 0.019450800915331808,
 'false_negative_rate': 0.7301587301587301}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

The model achieves a misclassification rate of about 10.7% on the test set, which means it correctly identifies around 89% of accounts. However, I have moderate confidence in using this model for real bot detection. The main concern is that the model performs much better on the training data than the test data, suggesting it may have memorized patterns from the training set rather than learning generalizable rules. More importantly, the model appears to miss a significant number of actual bots (high false negative rate), which is problematic since the whole point is to catch bots. While the overall accuracy looks decent, the model would need more work to better identify bots and should be tested on more data before being used in practice.

### What are potential ramifications of false positives from the model?

False positives happen when the model incorrectly labels a real person's account as a bot. This creates several problems. First, legitimate users who get flagged may have their accounts restricted or even banned, which is very frustrating and makes them not want to use the platform anymore. If these users complain publicly about being wrongly flagged, it can hurt the platform's reputation and make others hesitant to join. Secondly, false positives create extra work for customer service teams who have to review appeals and manually check accounts. 

### What are potential ramifications of false negatives from the model?

False negatives occur when the model fails to catch actual bots, letting them continue operating on the platform. This is a serious problem because undetected bots can cause significant harm. They can spam users, spread false information, and work together to manipulate what people see and think. Bots can also cheat the system by artificially boosting likes, followers, or views, which makes the platform's metrics unreliable. From a business standpoint, this means companies might make bad decisions based on fake engagement numbers. Additionally, if platforms don't catch enough bots, especially during important events like elections or public health crises, they could face legal trouble. Since this model has a high false negative rate, it would miss most bots, making the detection system not very useful in practice but maybe can be used as a way to measure risk instead if decision making.